# Warframe stuff & data

answers the ff. questions:
1. whats the most expensive arcane in warframe.market
   - list of arcanes
   - call warframe.market api w/ timeouts to follow rate limit
   - get a list of prices from warframe.market
   - maybe use sqlite to cache request results

- resources
  - [warframe.market api docs](https://42bytes.notion.site/WFM-Api-v2-Documentation-5d987e4aa2f74b55a80db1a09932459d)

In [31]:
import requests
import pandas as pd
import sqlite3
from ratelimit import limits, sleep_and_retry
from datetime import datetime, timezone, timedelta

WARFRAME_MARKET_API = "https://api.warframe.market/v2"
WARFRAME_MARKET_API_V1 = "https://api.warframe.market/v1"


def get_items():
    return requests.get(f"{WARFRAME_MARKET_API}/items").json()['data']

@sleep_and_retry
@limits(calls=10, period=1)
def get_stats(item):
    return requests.get(f"{WARFRAME_MARKET_API_V1}/items/{item}/statistics").json()[
        "payload"
    ]

1. setup table

In [32]:
with sqlite3.connect("./assets/data.db") as conn:
    c = conn.cursor()

    c.execute("DROP TABLE IF EXISTS arcanes;")
    c.execute("DROP TABLE IF EXISTS statistics;")

    c.execute("PRAGMA foreign_keys = 1")
    c.execute("""
        CREATE TABLE IF NOT EXISTS arcanes (
            name VARCHAR(255) PRIMARY KEY,
            max_rank int,
            rarity VARCHAR(255),
            type VARCHAR(255)
        );
    """)

    c.execute("""
        CREATE TABLE IF NOT EXISTS statistics (
            item_name VARCHAR(255) NOT NULL,
            datetime DATETIME,
            volume INT,
            min_price INT,
            max_price INT,
            open_price INT,
            closed_price INT,
            mod_rank INT,
            FOREIGN KEY (item_name) REFERENCES arcanes(name),
            PRIMARY KEY (datetime, mod_rank, item_name)
        );
    """)

    conn.commit()

2. setup arcane data

In [33]:
def convert_to_slug(name: str):
    sep = "-" if "-" in name else "_"
    return name.replace(" ", sep).lower()


with sqlite3.connect("./assets/data.db") as conn:
    res = pd.read_json("./assets/arcanes.json", orient="index")
    res[["Name", "MaxRank", "Rarity", "Type"]].rename(
        columns={
            "Name": "name",
            "MaxRank": "max_rank",
            "Rarity": "rarity",
            "Type": "type",
        }
    ).assign(name=lambda df: df["name"].map(convert_to_slug)).to_sql(
        "arcanes", conn, if_exists="append", index=False
    )

3. pull arcane statistics

In [34]:
with sqlite3.connect("./assets/data.db") as conn:
    res = pd.read_sql_query("SELECT name FROM arcanes;", conn)
    c = conn.cursor()
    c.execute("PRAGMA foreign_keys = ON")

    for i, arcane in res.iterrows():
        data = get_stats(arcane["name"])
        statistics_closed = pd.DataFrame(data["statistics_closed"]["90days"])
        df = statistics_closed[
            [
                "datetime",
                "volume",
                "min_price",
                "max_price",
                "open_price",
                "closed_price",
                "mod_rank",
            ]
        ].assign(item_name=arcane["name"])
        query = """
        INSERT INTO statistics (datetime, volume, min_price, max_price, open_price, closed_price, mod_rank, item_name)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT(datetime, mod_rank, item_name) DO UPDATE SET
            volume=excluded.volume,
            min_price=excluded.min_price,
            max_price=excluded.max_price,
            open_price=excluded.open_price,
            closed_price=excluded.closed_price
        """
        c.executemany(
            query,
            df[
                [
                    "datetime",
                    "volume",
                    "min_price",
                    "max_price",
                    "open_price",
                    "closed_price",
                    "mod_rank",
                    "item_name",
                ]
            ].itertuples(index=False, name=None),
        )

In [38]:
with sqlite3.connect("./assets/data.db") as conn:
    res = pd.read_sql_query("SELECT * FROM statistics;", conn)

    now = (datetime.now(timezone.utc) - timedelta(days=1)).replace(hour=0, minute=0, second=0, microsecond=0).isoformat(timespec='milliseconds')
    display(res.query(f"datetime == '{now}' & mod_rank == 5 & volume > 90").sort_values(['closed_price', 'volume'], ascending=False).head(20))

,item_name,datetime,volume,min_price,max_price,open_price,closed_price,mod_rank
4665,arcane_hot_shot,2026-07-02T00:00:00.000+00:00,101,400.0,455.0,440.0,415.0,5
8737,arcane_velocity,2026-07-02T00:00:00.000+00:00,96,100.0,130.0,120.0,118.0,5
511,arcane_aegis,2026-07-02T00:00:00.000+00:00,120,78.0,120.0,100.0,110.0,5
4050,arcane_fury,2026-07-02T00:00:00.000+00:00,98,67.0,100.0,96.0,95.0,5
3396,arcane_energize,2026-07-02T00:00:00.000+00:00,139,80.0,120.0,100.0,90.0,5
15359,melee_influence,2026-07-02T00:00:00.000+00:00,146,60.0,80.0,79.0,80.0,5
15894,molt_augmented,2026-07-02T00:00:00.000+00:00,139,48.0,52.0,50.0,50.0,5
2679,arcane_concentration,2026-07-02T00:00:00.000+00:00,140,27.0,30.0,30.0,30.0,5
1706,arcane_bellicose,2026-07-02T00:00:00.000+00:00,124,20.0,40.0,30.0,30.0,5
5708,arcane_persistence,2026-07-02T00:00:00.000+00:00,100,24.0,34.0,25.0,30.0,5
